## Final E5 training
Train the cleaned SQLite corpus as separate pair and triplet datasets. This keeps the optional hard-negative column optional while allowing an efficient no-duplicates batch sampler.

In [ ]:
import os
from pathlib import Path

# Kaggle T4 x2 defaults to DataParallel; use one GPU in this single-process notebook.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

BASE_MODEL_NAME = "intfloat/multilingual-e5-base"
OUTPUT_DIR = "/kaggle/working/VietRAG"
TRAIN_TABLE = "triplet"

BATCH_SIZE = 24
EPOCHS = 2
SEED = 42

LOCAL_DATA_PATH = Path("/home/nhminh/AI_Project/VietRAG-Embed-E5-Base/database/triplet_clean.db")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
kaggle_matches = sorted(KAGGLE_INPUT_ROOT.glob("**/triplet_clean.db")) if KAGGLE_INPUT_ROOT.exists() else []

if len(kaggle_matches) == 1:
    DATA_PATH = kaggle_matches[0]
elif len(kaggle_matches) > 1:
    raise RuntimeError(f"Found multiple triplet_clean.db files in Kaggle input: {kaggle_matches}")
elif LOCAL_DATA_PATH.exists():
    DATA_PATH = LOCAL_DATA_PATH
else:
    raise FileNotFoundError("Attach triplet_clean.db as a Kaggle input dataset before training.")

CACHE_DIR = Path("/kaggle/temp/vietrag_arrow_cache") if Path("/kaggle").exists() else Path("/tmp/vietrag_arrow_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Training database: {DATA_PATH}")

In [ ]:
import math
import sqlite3

from datasets import Dataset, DatasetDict, Features, Value

PAIR_FEATURES = Features({"anchor": Value("string"), "positive": Value("string")})
TRIPLET_FEATURES = Features(
    {
        "anchor": Value("string"),
        "positive": Value("string"),
        "hard_negative": Value("string"),
    }
)


def read_table_schema(connection: sqlite3.Connection) -> set[str]:
    return {row[1] for row in connection.execute(f"PRAGMA table_info({TRAIN_TABLE})")}


def load_sql_dataset(sql: str, features: Features, cache_name: str) -> Dataset:
    return Dataset.from_sql(
        sql=sql,
        con=f"sqlite:///{DATA_PATH}",
        features=features,
        cache_dir=str(CACHE_DIR / cache_name),
        keep_in_memory=False,
    )


with sqlite3.connect(DATA_PATH) as connection:
    columns = read_table_schema(connection)
    required_columns = {"anchor", "positive"}
    missing_columns = required_columns - columns
    if missing_columns:
        raise ValueError(f"{TRAIN_TABLE} is missing required columns: {sorted(missing_columns)}")

    valid_pair_sql = "anchor IS NOT NULL AND trim(anchor) <> '' AND positive IS NOT NULL AND trim(positive) <> ''"
    has_hard_negative = "hard_negative" in columns

    if has_hard_negative:
        triplet_count = connection.execute(
            f"SELECT COUNT(*) FROM {TRAIN_TABLE} WHERE {valid_pair_sql} AND hard_negative IS NOT NULL AND trim(hard_negative) <> ''"
        ).fetchone()[0]
        pair_count = connection.execute(
            f"SELECT COUNT(*) FROM {TRAIN_TABLE} WHERE {valid_pair_sql} AND (hard_negative IS NULL OR trim(hard_negative) = '')"
        ).fetchone()[0]
    else:
        triplet_count = 0
        pair_count = connection.execute(
            f"SELECT COUNT(*) FROM {TRAIN_TABLE} WHERE {valid_pair_sql}"
        ).fetchone()[0]

datasets_by_name = {}
if triplet_count:
    datasets_by_name["triplet"] = load_sql_dataset(
        f"SELECT anchor, positive, hard_negative FROM {TRAIN_TABLE} WHERE {valid_pair_sql} AND hard_negative IS NOT NULL AND trim(hard_negative) <> ''",
        TRIPLET_FEATURES,
        "triplets",
    )
if pair_count:
    datasets_by_name["pair"] = load_sql_dataset(
        f"SELECT anchor, positive FROM {TRAIN_TABLE} WHERE {valid_pair_sql} AND (hard_negative IS NULL OR trim(hard_negative) = '')"
        if has_hard_negative
        else f"SELECT anchor, positive FROM {TRAIN_TABLE} WHERE {valid_pair_sql}",
        PAIR_FEATURES,
        "pairs",
    )

if not datasets_by_name:
    raise ValueError("No valid pair or triplet rows were found.")

records_per_epoch = triplet_count + pair_count
steps_per_epoch = sum(math.ceil(len(dataset) / BATCH_SIZE) for dataset in datasets_by_name.values())
max_steps = steps_per_epoch * EPOCHS

train_dataset = (
    next(iter(datasets_by_name.values()))
    if len(datasets_by_name) == 1
    else DatasetDict(datasets_by_name)
)
print(
    f"Pairs: {pair_count:,}; triplets: {triplet_count:,}; records/epoch: {records_per_epoch:,}; "
    f"steps/epoch: {steps_per_epoch:,}; total steps: {max_steps:,}"
)

In [ ]:
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.sentence_transformer.training_args import (
    BatchSamplers,
    MultiDatasetBatchSamplers,
)

model = SentenceTransformer(BASE_MODEL_NAME)
loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=max_steps,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    fp16=True,
    seed=SEED,
    prompts={
        "anchor": "query: ",
        "positive": "passage: ",
        "hard_negative": "passage: ",
    },
    batch_sampler=BatchSamplers.NO_DUPLICATES_HASHED,
    multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
    dataloader_num_workers=2,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    logging_steps=100,
    save_strategy="steps",
    save_steps=5_000,
    save_total_limit=2,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()

In [ ]:
FINAL_MODEL_DIR = Path(OUTPUT_DIR) / "final"
trainer.save_model(str(FINAL_MODEL_DIR))
trainer.save_state()
print(f"Saved final SentenceTransformer model to: {FINAL_MODEL_DIR}")